# Token Classification using DistilBERT (POS Tagging & Chunking)
### Data Science Internship – February 2026
### Name: Hera Ansari
### Intern ID: IN226046402

In [7]:
!pip install transformers datasets seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=ebdcb35726810508f662946cbbe121cbacda437d8f94223f1abc5c9d6c807cb1
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


## Import

In [1]:
!pip install -U datasets

In [2]:
import numpy as np
import torch

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import TrainingArguments, Trainer

In [4]:
!pip install -U datasets huggingface_hub

In [5]:
import datasets
print(datasets.__version__)

4.8.4


# Load Dataset

## Task 1: Dataset Selection

Dataset: WikiANN

The WikiANN dataset is a multilingual dataset used for Named Entity Recognition (NER), which is a type of token classification task.

### Label Categories:
- PER (Person)
- ORG (Organization)
- LOC (Location)
- O (Other)

This dataset is suitable for sequence labeling tasks using transformer models.

In [6]:
from datasets import load_dataset

dataset = load_dataset("wikiann", "en")

dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  "\nThe secret `HF_TOKEN` does not exist in your Colab secrets."


README.md: 0.00B [00:00, ?B/s]

en/validation-00000-of-00001.parquet:   0%|          | 0.00/748k [00:00<?, ?B/s]

en/test-00000-of-00001.parquet:   0%|          | 0.00/748k [00:00<?, ?B/s]

en/train-00000-of-00001.parquet:   0%|          | 0.00/1.50M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

DatasetDict({
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 20000
    })
})

## Task 2: Data Preprocessing

The dataset was tokenized using the DistilBERT tokenizer.

### Steps performed:
- Tokenized input text using pre-trained tokenizer
- Used is_split_into_words=True for word-level mapping
- Aligned labels with tokens using word_ids()

### Handling:
- Subwords: Only first subword assigned label, rest set to -100
- Special tokens: Assigned -100 to ignore during training

### Output:
- input_ids
- attention_mask
- labels

In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [8]:
label_list = dataset["train"].features["ner_tags"].feature.names
num_labels = len(label_list)

print(label_list)

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']


In [9]:
def tokenize_and_align_labels(examples):

    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        padding=True
    )

    labels = []

    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)

        label_ids = []
        previous_word_idx = None

        for word_idx in word_ids:

            if word_idx is None:
                label_ids.append(-100)   # special tokens

            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])  # first subword

            else:
                label_ids.append(-100)  # other subwords

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [10]:
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

## Task 3: Model Setup

We used DistilBERT for token classification.

### Model:
- distilbert-base-uncased

### Implementation:
- Used AutoModelForTokenClassification
- Set num_labels based on dataset
- Defined label2id and id2label mappings

This setup enables the model to perform sequence labeling for token classification tasks.

In [11]:
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

print(label2id)

{'O': 0, 'B-PER': 1, 'I-PER': 2, 'B-ORG': 3, 'I-ORG': 4, 'B-LOC': 5, 'I-LOC': 6}


In [12]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Task 4: Training

The model was trained using Hugging Face Trainer.

### Parameters:
- Learning Rate: 2e-5
- Batch Size: 16
- Epochs: 1 (for faster training)

The model learns to predict labels for each token in the sequence.

In [13]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,   # fast
    per_device_eval_batch_size=16,
    num_train_epochs=1,               # 🔥 keep 1 (fast)
    logging_steps=50
)

In [14]:
train_dataset = tokenized_datasets["train"].select(range(1000))
val_dataset = tokenized_datasets["validation"].select(range(200))

In [15]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [16]:
import torch
print(torch.cuda.is_available())

True


In [18]:
trainer.train()

Step,Training Loss
50,1.480325


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=63, training_loss=1.4220351718720936, metrics={'train_runtime': 16.903, 'train_samples_per_second': 59.161, 'train_steps_per_second': 3.727, 'total_flos': 21947621916000.0, 'train_loss': 1.4220351718720936, 'epoch': 1.0})

## Task 5: Evaluation

The model was evaluated using the seqeval library.

### Metrics:
- Precision
- Recall
- F1 Score

These metrics are suitable for sequence labeling tasks like POS tagging and chunking.

In [19]:
!pip install seqeval

In [20]:
predictions, labels, _ = trainer.predict(val_dataset)

In [21]:
import numpy as np

preds = np.argmax(predictions, axis=2)

In [22]:
true_labels = []
true_preds = []

for pred, lab in zip(preds, labels):

    temp_preds = []
    temp_labels = []

    for p, l in zip(pred, lab):
        if l != -100:
            temp_preds.append(label_list[p])
            temp_labels.append(label_list[l])

    true_preds.append(temp_preds)
    true_labels.append(temp_labels)

In [23]:
from seqeval.metrics import precision_score, recall_score, f1_score

print("Precision:", precision_score(true_labels, true_preds))
print("Recall:", recall_score(true_labels, true_preds))
print("F1 Score:", f1_score(true_labels, true_preds))

Precision: 0.08904109589041095
Recall: 0.04727272727272727
F1 Score: 0.06175771971496437


## Task 6: Inference

The trained model was used to predict labels on custom sentences.

### Example:
Input: John works at Google in California

Output:
Each token is assigned a predicted label based on the trained model.

This demonstrates real-world usage of token classification models.

In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move model to device
model.to(device)

sentence = "John works at Google in California"
tokens = sentence.split()

# Tokenize
inputs = tokenizer(
    tokens,
    return_tensors="pt",
    is_split_into_words=True
)

# 🔥 Move inputs to GPU
inputs = {key: val.to(device) for key, val in inputs.items()}

# Prediction
outputs = model(**inputs)

preds = torch.argmax(outputs.logits, dim=2)

# Convert to labels
predicted_labels = [label_list[p.item()] for p in preds[0]]

print(list(zip(tokens, predicted_labels)))

[('John', 'O'), ('works', 'O'), ('at', 'O'), ('Google', 'O'), ('in', 'O'), ('California', 'O')]


## Task 7: Comparison

### POS Tagging:
- Assigns grammatical labels (noun, verb, adjective)
- Works at word level
- Easier task

### Chunking:
- Groups words into phrases (noun phrase, verb phrase)
- Works at phrase level
- More complex than POS tagging

### Conclusion:
Chunking is more complex than POS tagging as it captures structural relationships between words.

## Task 8: Report

### Differences:
- POS tagging labels individual words
- Chunking identifies meaningful phrases

### Challenges:
- Handling subword tokenization
- Aligning labels with tokens
- Managing special tokens

### Observations:
- Transformer models perform well for token classification
- Proper preprocessing improves performance

### Conclusion:
Fine-tuning transformer models like DistilBERT is effective for sequence labeling tasks such as POS tagging and chunking.

## Expected Pipeline Flow

The overall pipeline followed in this assignment is:

### 1. Raw Data
The dataset contains tokens (words) and corresponding labels (POS/NER tags).

### 2. Tokenization
The text is tokenized using a pre-trained transformer tokenizer (DistilBERT), converting words into input_ids and attention_mask.

### 3. Label Alignment
Labels are aligned with tokenized inputs using word_ids():
- First subword → assigned label
- Other subwords → assigned -100
- Special tokens → assigned -100

### 4. Model Training
A pre-trained DistilBERT model is fine-tuned using Hugging Face Trainer with defined hyperparameters.

### 5. Evaluation
The model is evaluated using seqeval metrics:
- Precision
- Recall
- F1 Score

### 6. Inference
The trained model is used to predict labels on custom input sentences.

### 7. Comparison
POS tagging and chunking are compared based on complexity and functionality.

### Conclusion
This pipeline demonstrates how transformer models can be used for token classification tasks effectively.